# Capstone Project — Design and Run a Claude Project for a Real Business Workflow

**Level: Intermediate**

## Objective

Bring together everything from Phases 1-5 to design, configure, and run a Claude Project that supports a real business workflow - a Marketing & Operations Assistant. You will configure instructions and knowledge, draft content, evaluate and validate outputs for accuracy and bias, apply a data-sensitivity redaction step, select the right model, and produce a final, review-ready deliverable.

### What You'll Build

A complete, runnable mini-workflow for a **Marketing/Ops Assistant** Project: a configuration, a drafting step, an evaluation + validation pass, a privacy redaction step, a model-selection rationale, and a final deliverable with a human-review sign-off.

### Rules

- Ground every decision in the exam domains (reference the lesson numbers given per step)
- The notebook must run end-to-end even without an API key (the ask_claude helper falls back to a mock)
- Never paste real regulated personal data - use fake sample data only
- Apply human review before the final deliverable is considered 'done'
- Verify any evolving Claude product/model detail at https://docs.anthropic.com

## Prerequisites

Before starting, make sure you have:

- [ ] Completed Phases 1-5
- [ ] The ask_claude helper from the setup cell (runs with or without an ANTHROPIC_API_KEY)
- [ ] A real-ish business scenario in mind (we use a monthly customer newsletter + trend summary)

**Estimated time:** 60-90 minutes

In [ ]:
# !pip install anthropic   # run once if the SDK is not installed
import os

def ask_claude(prompt, system=None, model="claude-3-5-haiku-latest", max_tokens=600):
    """Send a prompt to Claude. Falls back to a printed mock if no API key is set,
    so this notebook always runs. Set ANTHROPIC_API_KEY in your environment to use the real API.
    NOTE: Verify current model names at https://docs.anthropic.com (they change over time)."""
    if not os.environ.get("ANTHROPIC_API_KEY"):
        print("[No ANTHROPIC_API_KEY found - showing a MOCK response so the lesson still runs.]\n")
        return "[MOCK CLAUDE OUTPUT]\nThis is a placeholder. With a real key, Claude's answer appears here."
    try:
        import anthropic
        client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment
        kwargs = {"model": model, "max_tokens": max_tokens,
                  "messages": [{"role": "user", "content": prompt}]}
        if system:
            kwargs["system"] = system
        resp = client.messages.create(**kwargs)
        return resp.content[0].text
    except Exception as e:
        print(f"[Could not reach the API: {e}. Showing a mock response.]\n")
        return "[MOCK CLAUDE OUTPUT] Placeholder response."

print("Helper ready. Call ask_claude('your prompt here').")

## Step 1: Define the workflow and find the use case

**Lesson reference:** Lessons 4.1-4.2 (requirements & integration), 1.1 (surfaces)

Pick a real workflow. Here: a **monthly customer newsletter** plus a **customer-trend summary** for the ops lead. Document the goal, audience, inputs, outputs, and where a human must review.

In [ ]:
workflow = {
  'goal': 'Produce a monthly newsletter draft + a trend summary for the ops lead',
  'audience': 'Customers (newsletter) and internal ops lead (summary)',
  'inputs': 'Product updates, last month notes, a (fake) customer data extract',
  'outputs': 'Newsletter draft (artifact) + trend summary (structured)',
  'human_review': 'Required before anything is sent externally',
}
for k,v in workflow.items():
    print(f'{k}: {v}')

### Checkpoint 1: Verify Define the workflow and find the use case

Confirm the workflow has a clear goal, audience, inputs/outputs, and a human-review point.

In [ ]:
assert workflow['goal'] and workflow['human_review']
print('Checkpoint 1 passed: workflow scoped with a human-review point.')

## Step 2: Configure the Project: instructions + knowledge

**Lesson reference:** Lesson 4.4 (configuring Projects), 4.5 (maintenance)

Write system-level instructions and list the knowledge sources. Good instructions set role, tone, rules, format, and a guardrail to flag claims needing verification.

In [ ]:
instructions = [
  'Role: Marketing & Operations assistant for a small SaaS company.',
  'Tone: friendly, clear, no jargon.',
  'Always: flag any statistic or claim that needs human verification.',
  'Never: include unverified figures or regulated personal data.',
  'Format: newsletter as a short artifact; trend summary as bullet points.',
]
knowledge_sources = ['Brand voice guide', 'Product update log', 'Newsletter template', 'Anonymized trend data']
print('INSTRUCTIONS:')
for i in instructions: print(' -', i)
print('\nKNOWLEDGE SOURCES:')
for k in knowledge_sources: print(' -', k)

### Checkpoint 2: Verify Configure the Project: instructions + knowledge

Confirm instructions include a verification guardrail and a no-sensitive-data rule.

In [ ]:
joined = ' '.join(instructions).lower()
assert 'flag' in joined and 'regulated personal data' in joined
print('Checkpoint 2 passed: guardrails present in instructions.')

## Step 3: Apply the data-sensitivity redaction step

**Lesson reference:** Lesson 5.2 (data sensitivity) - Sample Question 3

Before any customer data is used, anonymize identifiers. This mirrors Sample Question 3: remove/anonymize regulated identifiers BEFORE uploading, consistent with policy. Uses FAKE data only.

In [ ]:
import re
def anonymize(text):
    text = re.sub(r'\b\d{6,}\b', '[ACCOUNT]', text)
    text = re.sub(r'\b[\w.]+@[\w.]+\b', '[EMAIL]', text)
    return text

fake_extract = 'Jane Doe, acct 100245678, jane@acme.com, upgraded plan. Bob Lee, acct 100987654, bob@acme.com, churned.'
clean_extract = anonymize(fake_extract)
print('BEFORE:', fake_extract)
print('AFTER :', clean_extract)

### Checkpoint 3: Verify Apply the data-sensitivity redaction step

Confirm no long account numbers or emails remain in the cleaned extract.

In [ ]:
import re
assert not re.search(r'\b\d{6,}\b', clean_extract)
assert '@' not in clean_extract
print('Checkpoint 3 passed: identifiers redacted before use.')

## Step 4: Select the right model for each sub-task

**Lesson reference:** Lessons 3.2-3.3 (models) - Sample Question 2

Match model to task. High-volume short drafts -> faster, lower-cost model; nuanced trend analysis -> more capable model. Don't default to the top model for everything (Sample Q2).

In [ ]:
def pick_model(complex_reasoning, high_volume, high_stakes):
    if complex_reasoning or high_stakes:
        return 'Opus-class + human review'
    if high_volume:
        return 'Haiku-class (fast/low-cost)'
    return 'Sonnet-class (balanced)'

choices = {
  'Many short newsletter blurbs': pick_model(False, True, False),
  'Nuanced customer-trend analysis': pick_model(True, False, True),
}
for task, model in choices.items():
    print(f'- {task} -> {model}')

### Checkpoint 4: Verify Select the right model for each sub-task

Confirm the high-volume task uses a faster model and the complex task escalates.

In [ ]:
assert 'Haiku' in choices['Many short newsletter blurbs']
assert 'Opus' in choices['Nuanced customer-trend analysis']
print('Checkpoint 4 passed: model choices aligned to cost/speed/quality.')

## Step 5: Draft the content (newsletter + trend summary)

**Lesson reference:** Lessons 1.2-1.5 (prompting), 2.6 (formats)

Use structured prompts to produce the two deliverables. The trend summary uses only the anonymized data.

In [ ]:
newsletter = ask_claude('Role: friendly SaaS marketer. Task: draft a 3-paragraph monthly newsletter about a new scheduling feature. Format: warm, plain language, end with a call to action.')
print('--- NEWSLETTER DRAFT ---\n', newsletter)
trend = ask_claude(f'Summarize customer trends from this ANONYMIZED data as 3 bullets for an ops lead. Flag anything to verify.\n\n{clean_extract}')
print('\n--- TREND SUMMARY ---\n', trend)

### Checkpoint 5: Verify Draft the content (newsletter + trend summary)

Confirm both deliverables were produced (non-empty).

In [ ]:
assert newsletter and trend
print('Checkpoint 5 passed: both deliverables drafted.')

## Step 6: Evaluate and validate the outputs

**Lesson reference:** Lessons 2.1-2.5 (evaluation, hallucinations, bias, fact-checking, human review) - Sample Question 1

Run the evaluation checklist, scan for fabricated specifics, check for bias, and record fact-checks. Remember Sample Q1: verify any specific claim/citation against an authoritative source; self-reported confidence is NOT a reliability signal.

In [ ]:
evaluation = {
  'Answers the request': 'TODO Y/N',
  'Complete (all parts present)': 'TODO Y/N',
  'Any fabricated stats/citations?': 'TODO - list and verify against source',
  'Bias / one-sidedness?': 'TODO - note any',
  'Claims needing human verification': 'TODO - list',
}
print('EVALUATION (fill in):')
for k,v in evaluation.items():
    print(f'  - {k}: {v}')
print('\nReminder: verify specifics against the authoritative source; confidence is not proof.')

### Checkpoint 6: Verify Evaluate and validate the outputs

Confirm you have identified which claims (if any) require verification before sharing.

In [ ]:
assert 'Claims needing human verification' in evaluation
print('Checkpoint 6 passed: validation pass defined; verify flagged claims before sharing.')

## Step 7: Produce the final deliverable with human-review sign-off

**Lesson reference:** Lessons 2.5 (human review), 4.3 (communicating value/limits)

Assemble the final package and record a human-review sign-off. Save it to a file as your capstone artifact.

In [ ]:
reviewer = 'TODO: your name'
sign_off = 'TODO: Approved / Changes needed'
final = (f'# Monthly Marketing & Ops Package\n\n'
         f'## Newsletter draft\n{newsletter}\n\n'
         f'## Trend summary (from anonymized data)\n{trend}\n\n'
         f'## Human review\nReviewer: {reviewer}\nDecision: {sign_off}\n')
with open('capstone_final_package.md','w',encoding='utf-8') as f:
    f.write(final)
print('Saved capstone_final_package.md')
print(final[:400], '...')

### Checkpoint 7: Verify Produce the final deliverable with human-review sign-off

Confirm the final package file was written and includes a human-review section.

In [ ]:
import os
assert os.path.exists('capstone_final_package.md')
with open('capstone_final_package.md', encoding='utf-8') as f:
    assert 'Human review' in f.read()
print('Checkpoint 7 passed: final package saved with human-review sign-off.')

## Verification Checklist

Confirm each item before considering the capstone complete:

- [ ] Workflow scoped with goal, audience, inputs/outputs, and a human-review point (L4.1-4.2)
- [ ] Project instructions include a verification guardrail and no-sensitive-data rule (L4.4)
- [ ] Knowledge sources listed and sensitive data anonymized before use (L5.2 / Sample Q3)
- [ ] Model choices match cost/speed/quality per sub-task (L3.2-3.3 / Sample Q2)
- [ ] Both deliverables drafted with structured prompts (L1.2-1.5)
- [ ] Outputs evaluated for accuracy, completeness, hallucinations, and bias (L2.1-2.4)
- [ ] Specific claims flagged and verified against an authoritative source (L2.4 / Sample Q1)
- [ ] Final package saved with a documented human-review sign-off (L2.5)

## Stretch Goals

Completed the core project? Try these extensions:

1. **Add a second audience version** — Produce an executive-brief version of the trend summary (Lesson 2.6) and compare.
2. **Build a maintenance schedule** — Create a review cadence for the Project's instructions and knowledge (Lesson 4.5).
3. **Write a stakeholder briefing** — Summarize the Project's value and limitations for a manager (Lesson 4.3).
4. **Optimize and measure** — Template the prompts, estimate time saved, and record a before/after metric (Lesson 5.5).
5. **Draft an org-aligned use pledge** — Add a responsible-use pledge covering appropriate use and ethics (Lessons 5.1, 5.3).

---

## Congratulations! 🎉

You designed and ran a complete, responsible Claude workflow end-to-end - configuring a Project, drafting content, validating it, protecting sensitive data, choosing the right model, and delivering a review-ready package. That is exactly the applied judgment the Claude Certified Associate - Foundations exam measures.

You've demonstrated proficiency in:

- Prompting and task decomposition (Domain 1)
- Output evaluation, validation, and bias/hallucination detection (Domain 2)
- Product feature and model selection for cost/speed/quality (Domain 3)
- Workflow integration and solution design (Domain 4)
- Project configuration and knowledge management (Domain 5)
- Governance, data sensitivity, and responsible use (Domain 6)
- Troubleshooting and workflow optimization (Domain 7)

**What's next?**

- Review any lessons you found challenging
- Attempt the stretch goals above
- Book your exam when you can complete this capstone unaided